# # 04 - Model 3: Denial Reason Boosting (multiclass)
#
# **Objective:** for a claim that is at risk of denial, predict the **most likely
# denial reason category** so the billing team knows what to fix before submission.
#
# * Target: `denial_category` (7 classes, denied claims only).
# * Algorithm: **XGBoost multiclass** classifier inside a Scikit-learn pipeline.
# * The actual `denial_reason_code` is **never** used as an input feature (it would leak
#   the answer). Features are exactly the **same 5 selected features** as Models 1 & 2.
# * Rare classes: the smallest class has 1,692 rows, so all 7 are kept.
# * Tuning: **RandomizedSearchCV (n_iter=10)**, 3-fold stratified CV, `scoring="f1_macro"`.


# ## 1. Imports & paths


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)
from xgboost import XGBClassifier

RANDOM_STATE = 42
BASE = Path.cwd()
while not (BASE / "data" / "processed").exists() and BASE != BASE.parent:
    BASE = BASE.parent
PROCESSED = BASE / "data" / "processed"
MODELS = BASE / "models"
REPORTS = BASE / "reports"
MODELS.mkdir(exist_ok=True)
REPORTS.mkdir(exist_ok=True)


# ## 2. Load & filter to denied claims


In [2]:
df = pd.read_csv(PROCESSED / "claims_model_ready.csv")
with open(PROCESSED / "feature_columns.json", encoding="utf-8") as fh:
    cols = json.load(fh)
# Only the 5 features selected by the feature-selection step in notebook 01.
CATEGORICAL, NUMERIC, FEATURES = (cols["selected_categorical"],
                                  cols["selected_numeric"], cols["selected_features"])
print("Selected features (%d):" % len(FEATURES), FEATURES)

data = df[df["denial_category"].notna()].copy()
print("Denied claims:", data.shape)
print(data["denial_category"].value_counts())


Selected features (5): ['documentation_completeness', 'cpt_code', 'primary_icd10_dx', 'claim_amount_usd', 'auth_status']
Denied claims: (33664, 40)
denial_category
medical_necessity    9444
coding_error         7344
auth_missing         6140
eligibility          4084
duplicate            2645
timely_filing        2315
bundling             1692
Name: count, dtype: int64


# ## 3. Encode target & split


In [3]:
le = LabelEncoder()
data["y"] = le.fit_transform(data["denial_category"])
classes = list(le.classes_)
print("Classes:", classes)

X = data[FEATURES]
y = data["y"]
X_train, y_train = X[data["split"] == "train"], y[data["split"] == "train"]
X_val, y_val = X[data["split"] == "val"], y[data["split"] == "val"]
X_test, y_test = X[data["split"] == "test"], y[data["split"] == "test"]
print("train/val/test:", X_train.shape, X_val.shape, X_test.shape)


Classes: ['auth_missing', 'bundling', 'coding_error', 'duplicate', 'eligibility', 'medical_necessity', 'timely_filing']


train/val/test: (23597, 5) (4965, 5) (5102, 5)


# ## 4. Preprocessing pipeline


In [4]:
numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                         ("scaler", StandardScaler())])
categorical_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                             ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
preprocessor = ColumnTransformer([("num", numeric_pipe, NUMERIC),
                                  ("cat", categorical_pipe, CATEGORICAL)])


# ## 5. Hyper-parameter search (RandomizedSearchCV, n_iter = 10)


In [5]:
pipe = Pipeline([("prep", preprocessor),
                 ("clf", XGBClassifier(objective="multi:softprob", num_class=len(classes),
                                       tree_method="hist", eval_metric="mlogloss",
                                       random_state=RANDOM_STATE, n_jobs=1))])

param_dist = {
    "clf__n_estimators": [200, 300, 400, 600],
    "clf__max_depth": [3, 4, 5, 6, 8],
    "clf__learning_rate": [0.03, 0.05, 0.1, 0.2],
    "clf__subsample": [0.7, 0.8, 0.9, 1.0],
    "clf__colsample_bytree": [0.6, 0.7, 0.8, 0.9],
    "clf__min_child_weight": [1, 3, 5],
    "clf__gamma": [0, 0.1, 0.3],
    "clf__reg_alpha": [0, 0.1, 1],
    "clf__reg_lambda": [1, 1.5, 2],
}
search = RandomizedSearchCV(pipe, param_dist, n_iter=10, scoring="f1_macro",
                            cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
                            random_state=RANDOM_STATE, n_jobs=-1, refit=True)
search.fit(X_train, y_train)
print("Best CV macro-F1:", round(search.best_score_, 4))
print("Best params      :", search.best_params_)


Best CV macro-F1: 0.209
Best params      : {'clf__subsample': 1.0, 'clf__reg_lambda': 2, 'clf__reg_alpha': 1, 'clf__n_estimators': 300, 'clf__min_child_weight': 3, 'clf__max_depth': 8, 'clf__learning_rate': 0.2, 'clf__gamma': 0, 'clf__colsample_bytree': 0.8}


# ## 6. Evaluate on the untouched test set


In [6]:
best = search.best_estimator_
pred = best.predict(X_test)
metrics = {
    "accuracy": round(accuracy_score(y_test, pred), 4),
    "precision_macro": round(precision_score(y_test, pred, average="macro", zero_division=0), 4),
    "recall_macro": round(recall_score(y_test, pred, average="macro", zero_division=0), 4),
    "macro_f1": round(f1_score(y_test, pred, average="macro", zero_division=0), 4),
    "weighted_f1": round(f1_score(y_test, pred, average="weighted", zero_division=0), 4),
    "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
    "classes": classes,
    "test_rows": int(len(y_test)),
}
print(json.dumps(metrics, indent=2))
print(classification_report(y_test, pred, target_names=classes, zero_division=0))


{
  "accuracy": 0.3503,
  "precision_macro": 0.2215,
  "recall_macro": 0.2266,
  "macro_f1": 0.2057,
  "weighted_f1": 0.3041,
  "confusion_matrix": [
    [
      507,
      3,
      204,
      6,
      20,
      203,
      2
    ],
    [
      34,
      0,
      60,
      4,
      11,
      140,
      2
    ],
    [
      217,
      4,
      412,
      18,
      46,
      429,
      6
    ],
    [
      53,
      2,
      74,
      13,
      28,
      224,
      9
    ],
    [
      73,
      8,
      122,
      15,
      35,
      331,
      11
    ],
    [
      185,
      11,
      286,
      25,
      73,
      814,
      14
    ],
    [
      47,
      2,
      72,
      8,
      21,
      212,
      6
    ]
  ],
  "classes": [
    "auth_missing",
    "bundling",
    "coding_error",
    "duplicate",
    "eligibility",
    "medical_necessity",
    "timely_filing"
  ],
  "test_rows": 5102
}
                   precision    recall  f1-score   support

     auth_missing       0.45     

# ## 7. Confusion matrix


In [7]:
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Purples",
            xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Denial reason confusion matrix")
plt.xticks(rotation=35, ha="right"); plt.yticks(rotation=0)
plt.tight_layout(); plt.savefig(REPORTS / "04_confusion_matrix.png", dpi=110); plt.show()


C:\Windows\Temp\ipykernel_30172\3405627917.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(REPORTS / "04_confusion_matrix.png", dpi=110); plt.show()


# ## 8. Feature importance


In [8]:
feat_names = best.named_steps["prep"].get_feature_names_out()
imp_df = (pd.DataFrame({"feature": feat_names,
                        "importance": best.named_steps["clf"].feature_importances_})
          .sort_values("importance", ascending=False).reset_index(drop=True))
plt.figure(figsize=(8, 6))
sns.barplot(data=imp_df.head(20), y="feature", x="importance")
plt.title("Denial reason model - top 20 features")
plt.tight_layout(); plt.savefig(REPORTS / "04_feature_importance.png", dpi=110); plt.show()
print(imp_df.head(15))


                            feature  importance
0     cat__auth_status_not_obtained    0.105595
1         cat__auth_status_obtained    0.048194
2     cat__auth_status_not_required    0.032269
3   num__documentation_completeness    0.024434
4     cat__primary_icd10_dx_S72.001    0.013085
5       cat__primary_icd10_dx_A41.9    0.012750
6         cat__primary_icd10_dx_I10    0.012194
7       cat__primary_icd10_dx_M54.5    0.011933
8      cat__primary_icd10_dx_Z34.00    0.011895
9               cat__cpt_code_95886    0.011853
10      cat__primary_icd10_dx_J18.9    0.011686
11     cat__primary_icd10_dx_C34.10    0.011528
12      cat__primary_icd10_dx_I50.9    0.011478
13     cat__primary_icd10_dx_I25.10    0.011271
14      cat__primary_icd10_dx_J44.1    0.011204


C:\Windows\Temp\ipykernel_30172\2137255755.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(REPORTS / "04_feature_importance.png", dpi=110); plt.show()


# ## 9. Save model + metadata
#
# The saved object is a dict so that the API can map predicted indices back to the
# class labels. Predictions are only meaningful for claims flagged as denial-risk;
# for low-risk claims the API reports `null` with an explanation.


In [9]:
joblib.dump({"pipeline": best, "classes": classes}, MODELS / "denial_reason_model.pkl")
meta_path = MODELS / "model_metadata.json"
meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
meta["denial_reason"] = {
    "algorithm": "XGBoost (multi:softprob)",
    "target": "denial_category",
    "best_params": {k: (v if not isinstance(v, (np.integer, np.floating)) else v.item())
                    for k, v in search.best_params_.items()},
    "cv_best_macro_f1": round(float(search.best_score_), 4),
    "metrics": metrics,
    "classes": classes,
    "features_used": FEATURES,
    "top_features": imp_df.head(15)["feature"].tolist(),
    "n_train": int(len(X_train)), "n_test": int(len(X_test)),
}
meta_path.write_text(json.dumps(meta, indent=2))
print("Saved", MODELS / "denial_reason_model.pkl")
print("=== MODEL 3 COMPLETE ===")


Saved D:\CDAC\Projects\ML_Project_Final\medical-claim-denial-predictor\models\denial_reason_model.pkl
=== MODEL 3 COMPLETE ===
